# Minimum Variance Analysis for selected intervals of Juno Magnetometer data.

In [ ]:
import os
import re
import glob
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

from matplotlib.ticker import AutoMinorLocator

from JunoPy.mag import MagData, mva
from JunoPy.library import find_ticks_between

## 1. Specify epoch and identify the magnetometer file.

In [ ]:
start = dt.datetime.strptime('2017-338T00:57:41', '%Y-%jT%H:%M:%S')
end   = dt.datetime.strptime('2017-338T01:03:40', '%Y-%jT%H:%M:%S')
magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
print('Magnetometer file to plot - {:s}'.format(magfile))

## 2. Read and crop the magnetometer file (+/- 10 minutes from epoch)

In [ ]:
def read_and_crop(magfile, start, end):
    """Read and crop magnetometer data."""
    magdata = MagData(filename=magfile)
    start_cut = start - dt.timedelta(minutes=5)
    end_cut   = end   + dt.timedelta(minutes=5)
    magdata.df = magdata.df.loc[start_cut:end_cut]

    magdata_mva = MagData(filename=magfile)
    magdata_mva.df = magdata_mva.df.loc[start:end]
    magdata_mva = mva(magdata_mva)

    # print(magdata)
    return magdata, magdata_mva

magdata, magdata_mva = read_and_crop(magfile, start, end)

## 3. Create the MVA composite plot.

In [ ]:
def plot_mva_composite(magdata, magdata_mva, start, end, savefile=False):

    fig = plt.figure(figsize=(11, 8.5), constrained_layout=False, dpi=300)

    gs1 = fig.add_gridspec(
        nrows=4, ncols=1, top=0.99, bottom=0.55, left=0.15, right=0.7, hspace=0)
    ax_magdata = list()
    for i in range(4):
        ax_magdata.append(fig.add_subplot(gs1[i, 0]))

    gs2 = fig.add_gridspec(
        nrows=3, ncols=1, top=0.45, bottom=0.1, left=0.15, right=0.66, hspace=0)
    ax_mvaB = list()
    for i in range(3):
        ax_mvaB.append(fig.add_subplot(gs2[i, 0]))

    gs3 = fig.add_gridspec(
        nrows=2, ncols=1, top=0.45, bottom=0.1, left=0.67, right=0.9, hspace=0)
    ax_hodo = list()
    for i in range(2):
        ax_hodo.append(fig.add_subplot(gs3[i, 0]))

    gs4 = fig.add_gridspec(
        nrows=1, ncols=1, top=0.9, bottom=0.55, left=0.71, right=0.9)
    ax_text = fig.add_subplot(gs4[0, 0])

    # 1. Plot the mag data +/- 10 min from epoch. ------------------------------------
    ax_magdata = magdata.plot(
        vars=['Br', 'Btheta', 'Bphi', 'Bmag'], axs=ax_magdata)
    timearray, tickstring = find_ticks_between(magdata.df.index[0], magdata.df.index[-1])
    ylabels = ['B$_r$', 'B$_\\theta$', 'B$_\phi$', '|B|']
    for i, a in enumerate(ax_magdata):
        a.set_xticks([], [])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.set_ylabel(ylabels[i] + ' (nT)')
        a.set_xlim(magdata.df.index[0], magdata.df.index[-1])
        a.label_outer()
        if start and end:
            a.axvspan(start, end, alpha=0.2, color=[0.2, 0.2, 0.2])
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)


    # 2. Plot the MVA B vectors. -------------------------------------------------------
    ax_mvaB = magdata_mva.plot(vars=['B1', 'B2', 'B3'], axs=ax_mvaB)
    timearray2, tickstring2 = find_ticks_between(magdata_mva.df.index[0], magdata_mva.df.index[-1])
    ylabels = ['B$_1$', 'B$_2$', 'B$_3$']
    for i, a in enumerate(ax_mvaB):
        a.set_xticks([], [])
        a.set_xticks(timearray2)
        a.set_xticklabels(tickstring2, fontdict={'horizontalalignment':'right'})
        a.set_xlim(magdata_mva.df.index[0], magdata_mva.df.index[-1])
        a.set_ylabel(ylabels[i] + ' (nT)')
        a.label_outer()

    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)

    dy = (ax_mvaB[-1].get_ylim()[1] - ax_mvaB[-1].get_ylim()[0]) / 2
    
    ax_mvaB[0].set_ylim([magdata_mva.df['B1'].mean() - dy,
                         magdata_mva.df['B1'].mean() + dy])
    ax_mvaB[1].set_ylim([magdata_mva.df['B2'].mean() - dy,
                         magdata_mva.df['B2'].mean() + dy])
    
    # 3. Plot the hodograms. -----------------------------------------------------------
    combo = [['B3', 'B1'],
             ['B3', 'B2']]
    Bmax = np.max(np.array(
            [magdata_mva.df['B1'].abs().max(), 
             magdata_mva.df['B2'].abs().max(), 
             magdata_mva.df['B3'].abs().max()]))  * 1.01
    
    
    for i, a in enumerate(ax_hodo):
        
        a.scatter(magdata_mva.df[combo[i][0]][0], magdata_mva.df[combo[i][1]][0], marker='o', color='b', s=50, label='Start')
        a.scatter(magdata_mva.df[combo[i][0]][-1], magdata_mva.df[combo[i][1]][-1], marker='x', color='r', s=50, label='End')
        a.plot(magdata_mva.df[combo[i][0]], magdata_mva.df[combo[i][1]],
               color='k',lw=0.4)
        
        a.set_xlabel(combo[i][0] + ' (nT)')
        a.set_ylabel(combo[i][1] + ' (nT)')
        a.set_aspect('equal', adjustable='datalim')
        a.xaxis.set_minor_locator(AutoMinorLocator())
        a.yaxis.set_minor_locator(AutoMinorLocator())
        a.grid('on', which='minor', lw=0.5, ls='dotted')
        a.grid(color='k', ls='dashed', lw=0.25)
        a.yaxis.tick_right()
        a.yaxis.set_label_position("right")
        a.label_outer()
    
    ax_hodo[0].legend(loc='upper right')
    ax_hodo[0].set_xlabel('B$_3$ (nT)')
    ax_hodo[0].set_ylabel('B$_1$ (nT)')
    ax_hodo[1].set_xlabel('B$_3$ (nT)')
    ax_hodo[1].set_ylabel('B$_2$ (nT)')
        
    # 4. Add Text ----------------------------------------------------------------------
    ax_text.axis('off')

    date = dt.datetime.strptime(start.strftime('%Y-%j'), '%Y-%j')

    text = []
    text.append(dt.datetime.strftime(date, '%Y-%m-%d') + ', DOY={:s}'.format(dt.datetime.strftime(date, '%j')))
    text.append('\n\n')
    text.append('MVA Output:\n--------\n\nCartesian Eigenvectors:\n')
    text.append('x$_1$ = ({:.2f}, {:.2f}, {:.2f})\n'.format(magdata_mva.x1[0], magdata_mva.x1[1], magdata_mva.x1[2]))
    text.append('x$_2$ = ({:.2f}, {:.2f}, {:.2f})\n'.format(magdata_mva.x2[0], magdata_mva.x2[1], magdata_mva.x2[2]))
    text.append('x$_3$ = ({:.2f}, {:.2f}, {:.2f})\n\n'.format(magdata_mva.x3[0], magdata_mva.x3[1], magdata_mva.x3[2]))
    text.append('--------\nRatio of eigenvalues:\n')
    text.append('$\lambda_3$ / $\lambda_2$ = {:.2f}\n'.format(magdata_mva.eigval[2] / magdata_mva.eigval[1]))
    text.append('$\lambda_2$ / $\lambda_1$ = {:.2f}\n'.format(magdata_mva.eigval[1] / magdata_mva.eigval[0]))
    text = ''.join(text)
    ax_text.text(0.5, 0.5, text, horizontalalignment='center', verticalalignment='center',
                 bbox=dict(boxstyle="round",
                   ec=(0.9, 0.9, 0.9),
                   fc=(0.9, 0.9, 0.9),
                   ))
    
    if savefile:
        plotfile = 'out_mva/' + dt.datetime.strftime(start, '%Y-%jT%H:%M:%S')
        fig.savefig(plotfile)
        plt.close(fig)

In [ ]:
def plot_mva_composite_xyz(magdata, magdata_mva, start, end, savefile=False):

    fig = plt.figure(figsize=(11, 8.5), constrained_layout=False, dpi=300)

    gs1 = fig.add_gridspec(
        nrows=4, ncols=1, top=0.99, bottom=0.55, left=0.15, right=0.7, hspace=0)
    ax_magdata = list()
    for i in range(4):
        ax_magdata.append(fig.add_subplot(gs1[i, 0]))

    gs2 = fig.add_gridspec(
        nrows=3, ncols=1, top=0.45, bottom=0.1, left=0.15, right=0.66, hspace=0)
    ax_mvaB = list()
    for i in range(3):
        ax_mvaB.append(fig.add_subplot(gs2[i, 0]))

    gs3 = fig.add_gridspec(
        nrows=2, ncols=1, top=0.45, bottom=0.1, left=0.67, right=0.9, hspace=0)
    ax_hodo = list()
    for i in range(2):
        ax_hodo.append(fig.add_subplot(gs3[i, 0]))

    gs4 = fig.add_gridspec(
        nrows=1, ncols=1, top=0.9, bottom=0.55, left=0.71, right=0.9)
    ax_text = fig.add_subplot(gs4[0, 0])

    # 1. Plot the mag data +/- 10 min from epoch. ------------------------------------
    ax_magdata = magdata.plot(
        vars=['Bx', 'By', 'Bz', 'Bmag'], axs=ax_magdata)
    timearray, tickstring = find_ticks_between(magdata.df.index[0], magdata.df.index[-1])
    ylabels = ['Bx', 'By', 'Bz', '|B|']
    for i, a in enumerate(ax_magdata):
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.set_ylabel(ylabels[i] + ' (nT)')
        a.set_xlim(magdata.df.index[0], magdata.df.index[-1])
        a.label_outer()
        if start and end:
            a.axvspan(start, end, alpha=0.2, color=[0.2, 0.2, 0.2])
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)


    # 2. Plot the MVA B vectors. -------------------------------------------------------
    ax_mvaB = magdata_mva.plot(vars=['B1', 'B2', 'B3'], axs=ax_mvaB)
    timearray, tickstring = find_ticks_between(magdata_mva.df.index[0], magdata_mva.df.index[-1])
    ylabels = ['B$_1$', 'B$_2$', 'B$_3$']
    for i, a in enumerate(ax_mvaB):
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring, fontdict={'horizontalalignment':'right'})
        a.set_xlim(magdata_mva.df.index[0], magdata_mva.df.index[-1])
        a.set_ylabel(ylabels[i] + ' (nT)')
        a.label_outer()

    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    dy = (ax_mvaB[-1].get_ylim()[1] - ax_mvaB[-1].get_ylim()[0]) / 2
    
    ax_mvaB[0].set_ylim([magdata_mva.df['B1'].mean() - dy,
                         magdata_mva.df['B1'].mean() + dy])
    ax_mvaB[1].set_ylim([magdata_mva.df['B2'].mean() - dy,
                         magdata_mva.df['B2'].mean() + dy])
    
    # 3. Plot the hodograms. -----------------------------------------------------------
    combo = [['B3', 'B1'],
             ['B3', 'B2']]
    Bmax = np.max(np.array(
            [magdata_mva.df['B1'].abs().max(), 
             magdata_mva.df['B2'].abs().max(), 
             magdata_mva.df['B3'].abs().max()]))  * 1.01
    
    
    for i, a in enumerate(ax_hodo):
    
        a.plot(magdata_mva.df[combo[i][0]], magdata_mva.df[combo[i][1]],
               color='k',lw=0.4)
        a.plot(magdata_mva.df[combo[i][0]][0], magdata_mva.df[combo[i][1]][0], '*')
        
        a.set_xlabel(combo[i][0] + ' (nT)')
        a.set_ylabel(combo[i][1] + ' (nT)')
        a.set_aspect('equal', adjustable='datalim')
        a.xaxis.set_minor_locator(AutoMinorLocator())
        a.yaxis.set_minor_locator(AutoMinorLocator())
        a.grid('on', which='minor', lw=0.5, ls='dotted')
        a.grid(color='k', ls='dashed', lw=0.25)
        a.yaxis.tick_right()
        a.yaxis.set_label_position("right")
        a.label_outer()
        
    ax_hodo[0].set_xlabel('B$_3$ (nT)')
    ax_hodo[0].set_ylabel('B$_1$ (nT)')
    ax_hodo[1].set_xlabel('B$_3$ (nT)')
    ax_hodo[1].set_ylabel('B$_2$ (nT)')
        
    # 4. Add Text ----------------------------------------------------------------------
    ax_text.axis('off')

    date = dt.datetime.strptime(start.strftime('%Y-%j'), '%Y-%j')

    text = []
    text.append(dt.datetime.strftime(date, '%Y-%m-%d') + ', DOY={:s}'.format(dt.datetime.strftime(date, '%j')))
    text.append('\n\n')
    text.append('MVA Output:\n--------\n\nCartesian Eigenvectors:\n')
    text.append('x$_1$ = ({:.2f}, {:.2f}, {:.2f})\n'.format(magdata_mva.x1[0], magdata_mva.x1[1], magdata_mva.x1[2]))
    text.append('x$_2$ = ({:.2f}, {:.2f}, {:.2f})\n'.format(magdata_mva.x2[0], magdata_mva.x2[1], magdata_mva.x2[2]))
    text.append('x$_3$ = ({:.2f}, {:.2f}, {:.2f})\n\n'.format(magdata_mva.x3[0], magdata_mva.x3[1], magdata_mva.x3[2]))
    text.append('--------\nRatio of eigenvalues:\n')
    text.append('$\lambda_3$ / $\lambda_2$ = {:.2f}\n'.format(magdata_mva.eigval[2] / magdata_mva.eigval[1]))
    text.append('$\lambda_2$ / $\lambda_1$ = {:.2f}\n'.format(magdata_mva.eigval[1] / magdata_mva.eigval[0]))
    text = ''.join(text)
    ax_text.text(0.5, 0.5, text, horizontalalignment='center', verticalalignment='center',
                 bbox=dict(boxstyle="round",
                   ec=(0.9, 0.9, 0.9),
                   fc=(0.9, 0.9, 0.9),
                   ))
    
    if savefile:
        plotfile = 'out_mva/' + dt.datetime.strftime(start, '%Y-%jT%H:%M:%S')
        fig.savefig(plotfile)
        plt.close(fig)

In [ ]:
if not os.path.exists('out_mva'):
    os.mkdir('out_mva')
    
plot_mva_composite(magdata, magdata_mva, start, end, savefile=False)

## 4. Create composite plots for all saved epochs.

In [ ]:
def save_dates_csv(dates):
    """Save the list of epochs in a csv file with columns - start, end, duration"""
    with open('mva_epochs.csv', 'w') as f:
        f.write('Start,End,Duration,Polarity\n')
        for date in dates:
            f.write(','.join(date.split()[:2]))
            start = dt.datetime.strptime(date.split()[0], '%Y-%jT%H:%M:%S')
            end = dt.datetime.strptime(date.split()[1], '%Y-%jT%H:%M:%S')
            duration = end - start
            f.write(',{:.1f}'.format(duration.total_seconds()))
            f.write(',{:s}\n'.format(date.split()[-1]))
            

In [ ]:
dates = [
    '2016-234T02:57:50 2016-234T02:58:39 NP',
    '2016-236T04:06:35 2016-236T04:07:22 PN',
    '2016-287T13:43:35 2016-287T13:43:47 NP',
    '2016-289T13:15:30 2016-289T13:16:05 NP',
    '2016-290T18:47:32 2016-290T18:47:44 NP',
    '2016-339T06:22:55 2016-339T06:23:10 PN',
    '2016-339T15:36:50 2016-339T15:37:30 NP',
    '2016-343T16:37:20 2016-343T16:37:40 PN',
    '2017-024T07:15:20 2017-024T07:17:45 NP',
    '2017-029T17:26:30 2017-029T17:27:11 PN',
    '2017-236T20:21:15 2017-236T20:21:38 NP',
    '2017-279T19:20:45 2017-279T19:21:25 NP',
    '2017-338T01:49:57 2017-338T01:51:00 PN',
    '2017-345T20:15:20 2017-345T20:15:49 NP',
    '2018-031T19:31:43 2018-031T19:32:52 PN',
    '2018-034T03:37:47 2018-034T03:38:35 PN',    
    '2018-085T15:53:45 2018-085T15:54:05 NP', 
    '2018-191T10:21:35 2018-191T10:23:13 PN',    # Change times 
    '2018-244T20:15:53 2018-244T20:16:25 PN', 
    '2018-291T03:06:42 2018-291T03:08:09 PN',
    '2018-299T05:38:26 2018-299T05:39:29 PN', 
    '2018-300T10:51:35 2018-300T10:52:15 NP', 
    '2018-301T12:21:42 2018-301T12:21:53 NP', 
    '2018-351T19:05:15 2018-351T19:06:25 NP', 
    '2018-353T04:04:31 2018-353T04:04:50 PN',    # Change times 
    '2018-353T13:12:15 2018-353T13:12:40 NP',
    '2018-353T13:59:12 2018-353T13:59:40 PN',
    '2018-354T00:12:58 2018-354T00:13:16 PN',
    '2019-041T01:46:03 2019-041T01:46:17 NP',
    '2019-089T20:16:40 2019-089T20:17:00 NP',
    '2019-089T20:35:40 2019-089T20:36:36 PN',
    '2019-094T06:37:05 2019-094T06:37:23 PN',
    '2019-146T01:11:42 2019-146T01:12:20 PN',
    '2019-147T04:29:53 2019-147T04:31:11 PN'
]

save_dates_csv(dates)

In [ ]:
for date in dates:
    start = dt.datetime.strptime(date.split()[0], '%Y-%jT%H:%M:%S')
    end = dt.datetime.strptime(date.split()[-2], '%Y-%jT%H:%M:%S')
    magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
    magdata, magdata_mva = read_and_crop(magfile, start, end)
    plot_mva_composite(magdata, magdata_mva, start, end, savefile=True)

In [ ]:
## Histograms and statistics

In [ ]:
def plot_histogram(dates):
        
    # Calculate durations again
    durations = []
    for date in dates:
        start = dt.datetime.strptime(date.split()[0], '%Y-%jT%H:%M:%S')
        end = dt.datetime.strptime(date.split()[1], '%Y-%jT%H:%M:%S')
        durations.append((end - start).total_seconds())
        
    durations = np.asarray(durations)
    
#     bins_kronberg = np.array([0, 10, 20, 30, 40, 50]) * 60
#     counts_kronberg = np.array([6, 9, 5, 2, 1])
    
#     bins_vogt = np.arange(0, 32, 2) * 60
#     counts_vogt = np.array([3, 9, 11, 5, 8, 1, 2, 2, 0, 1, 0, 0, 1, 0, 0])
    
    # Create new figure and plot
    fig = plt.figure(dpi=600)
    ax = plt.subplot(111)
    bins = np.arange(0, 160, 5)
    ax.hist(durations, bins=bins, color=[0.5, 0.5, 0.5], ec='k', zorder=2)
    
#     ax.hist(bins_kronberg[:-1], bins=bins_kronberg, weights=counts_kronberg, 
#             color=[0.8, 0.5, 0.5], ec='k', zorder=1)
    
#     ax.hist(bins_vogt[:-1], bins=bins_vogt, weights=counts_vogt, 
#             color=[0.8, 0.5, 0.5], ec='k', zorder=1)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.set_xlim([0, 130])
    ax.set_xlabel('Duration (seconds)')
    ax.set_ylabel('Number of events')
    ax.text(120,5, 'N = {:d}\nMean - {:.2f} s\nMedian - {:.2f} s'.format(
                len(durations), np.mean(durations), np.median(durations)),
            horizontalalignment='right')
    plt.show()
    fig.savefig('out_images/hist_duration.png')
    plt.close(fig)
    
plot_histogram(dates)

## Create barstacked histogram

In [ ]:
def read_csv_file():
    durations = []
    PNevents = []
    NPevents = []
    with open('mva_epochs.csv', 'r') as f:
        f.readline()
        
        for i, line in enumerate(f):    
            durations.append(float(line.split(',')[2]))
            
            if 'PN' in line.split(',')[3]:
                PNevents.append(i)
            else:
                NPevents.append(i)
            
    return np.asarray(durations), PNevents, NPevents
        

def plot_barstacked_histogram(durations, PNevents, NPevents):
        
    
#     bins_kronberg = np.array([0, 10, 20, 30, 40, 50]) * 60
#     counts_kronberg = np.array([6, 9, 5, 2, 1])
    
#     bins_vogt = np.arange(0, 32, 2) * 60
#     counts_vogt = np.array([3, 9, 11, 5, 8, 1, 2, 2, 0, 1, 0, 0, 1, 0, 0])
    
    # Create new figure and plot
    fig = plt.figure(dpi=600)
    ax = plt.subplot(111)
    bins = np.arange(0, 160, 5)
    n, bins, patches = ax.hist([durations[PNevents], durations[NPevents]], 
                                bins=bins, 
                                histtype='barstacked',
                                ec='k',
                                color=[[0.5, 0.5, 0.5], [1.0, 1.0, 1.0]],
                                zorder=2)
    
#     hatches = ['', '...']
#     for patch_set, hatch in zip(patches, hatches):
#         for patch in patch_set.patches:
#             patch.set_hatch(hatch)
        
#     ax.hist(bins_kronberg[:-1], bins=bins_kronberg, weights=counts_kronberg, 
#             color=[0.8, 0.5, 0.5], ec='k', zorder=1)
    
#     ax.hist(bins_vogt[:-1], bins=bins_vogt, weights=counts_vogt, 
#             color=[0.8, 0.5, 0.5], ec='k', zorder=1)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    # ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.set_xlim([0, 130])
    ax.set_xlabel('Duration (seconds)')
    ax.set_ylabel('Number of events')
    ax.text(100, 3.5,
            'N = {:d}\n'.format(len(durations)) + 
            'Mean (total) - {:.2f} s\n'.format(np.mean(durations)) + 
            'Median (total) - {:.2f} s\n'.format(np.median(durations)) + 
            'Mean (+/-) - {:.2f} s\n'.format(np.mean(durations[PNevents])) + 
            'Median (+/-) - {:.2f} s\n'.format(np.median(durations[PNevents])) +
            'Mean (-/+) - {:.2f} s\n'.format(np.mean(durations[NPevents])) + 
            'Median (-/+) - {:.2f} s'.format(np.median(durations[NPevents])),
            horizontalalignment='right')
    
    ax.yaxis.set_ticks([0, 2, 4, 6, ])
    ax.legend(['+/-', '-/+'])
    plt.show()
    fig.savefig('out_images/hist_duration_different.png')
    plt.close(fig)
    
    
durations, PNevents, NPevents = read_csv_file()
plot_barstacked_histogram(durations, PNevents, NPevents)


In [ ]:
# Craete new csv
def save_dates_csv_new(dates):
    """Save the list of epochs in a csv file with columns - start, end, duration"""
    with open('mva_epochs_selected.csv', 'w') as f:
        f.write('Start,End,Duration,Polarity\n')
        for date in dates:
            f.write(','.join(date.split()[:2]))
            start = dt.datetime.strptime(date.split()[0], '%Y-%jT%H:%M:%S')
            end = dt.datetime.strptime(date.split()[1], '%Y-%jT%H:%M:%S')
            duration = end - start
            f.write(',{:.1f}'.format(duration.total_seconds()))
            f.write(',{:s}\n'.format(date.split()[-1]))
            

In [ ]:
new_dates = [
    '2016-234T02:57:50 2016-234T02:58:39 NP',
    '2016-236T04:06:35 2016-236T04:07:22 PN',
    '2016-339T06:22:55 2016-339T06:23:10 PN',
    '2016-343T16:37:20 2016-343T16:37:40 PN',
    '2017-024T07:15:20 2017-024T07:17:45 NP',
    '2017-236T20:21:15 2017-236T20:21:38 NP',
    '2017-338T01:49:57 2017-338T01:51:00 PN',
    '2018-034T03:37:47 2018-034T03:38:35 PN',    
    '2018-244T20:15:53 2018-244T20:16:25 PN', 
    '2018-299T05:38:26 2018-299T05:39:29 PN', 
    '2018-300T10:51:35 2018-300T10:52:15 NP', 
    '2018-301T12:21:42 2018-301T12:21:53 NP', 
    '2018-353T04:04:31 2018-353T04:04:50 PN',    # Change times 
    '2018-353T13:12:15 2018-353T13:12:40 NP',
    '2018-353T13:59:12 2018-353T13:59:40 PN',
    '2018-354T00:12:58 2018-354T00:13:16 PN',
    '2019-041T01:46:03 2019-041T01:46:17 NP',
    '2019-089T20:16:40 2019-089T20:17:00 NP',
    '2019-089T20:35:40 2019-089T20:36:36 PN',
    '2019-147T04:29:53 2019-147T04:31:11 PN'
]

save_dates_csv_new(new_dates)